**Project:** Data Mining II (2025/26)

**Group Number:** 12

**Members and Group Member Contribution:**
- Dinis Gaspar - 20221869 (40%)
    - Dataset Exploration
    - Additional Preprocessing
    - Modelling Methodology
    - Pipeline Development
    - Parameter Search
    - Interpretation of Results
    - Dashboard Development

- Margarida Cruz - 20221929 (30%)
    - Dataset Exploration
    - EDA Conclusions
    - Modelling Methodology
    - Pipeline Development
    - Interpretation of Results

- Beatriz Boura - 20250272 (30%)
    - Dataset Exploration
    - EDA Conclusions
    - Modelling Methodology
    - Interpretation of Results
    - Dashboard Development

**Abstract**

This project addresses the challenge faced by nonprofit fundraisers: how to move away from broad, repeated solicitation and toward a targeted donor outreach strategy. Using the Civic Support Alliance donor dataset, our goal was to build a predictive system that estimates whether a given individual will respond positively to a new campaign contact. The context is donor behavior analytics, with the aim of improving campaign efficiency while reducing donor fatigue.

Our work began with exploratory data analysis and careful data cleaning. We developed a reusable pipeline architecture in 01_Modeling_Tools.ipynb, including custom transformers for cleaning, outlier clipping, feature engineering, and feature selection. The pipeline handled identifier removal, malformed values, categorical validation, and encoded donor attributes robustly. We also engineered donor-specific metrics such as normalized gift ratios, lifespan-based frequency, and income-relative generosity to enrich the model’s predictive signal.

We then evaluated a range of classifiers and preprocessing workflows, including decision trees, KNN, Naive Bayes, random forest, gradient boosting, AdaBoost, and others, using stratified cross-validation and threshold tuning focused on F1 performance. We also generated submission artifacts for Kaggle validation, and the best-performing models achieved public F1 scores in the mid-0.43 range.

A ready-to-deploy pipeline was developed. By scoring donors on conversion probability, the Civic Support Alliance can shift from costly blanket solicitation to precision outreach focused on high-conversion segments (recently engaged donors with stable finances and strong relative generosity). This reduces outreach costs and donor fatigue while improving conversion rates and marketing ROI when paired with calibrated probabilities and cost-sensitive thresholds. Moreover, an interactive Charity/Donor Behavior Analysis Dashboard was designed, allowing for raw donor universes auditing and real-time response donor predictions.

**Project Overview**

In recent years, nonprofit organizations have faced a growing challenge: while charitable causes have multiplied, public tolerance for repeated, generic solicitations has significantly decreased, often leading to donor fatigue and long-term disengagement. To address this issue, the Civic Support Alliance (CSA)—a federation representing multiple humanitarian and social aid programs—seeks to modernize its fundraising strategy. Rather than launching blanket campaigns across their entire database, the organization aims to transition to a highly targeted approach. The goal is to maximize operational efficiency and maintain donor respect by contacting fewer, but more receptive, individuals. 

As data scientists, our team has been tasked with building a predictive machine learning system using historical demographic, interaction, and donation data accumulated from past campaigns. The primary objective is to accurately answer a fundamental question: Will this person donate if contacted? 

**Notebook Introduction**

In this notebook, we will develop Ensemble models, we will build boosting and bagging models where we will then perform parameter searches to try and maximize performances. This notebook is lightly different from others in that we jump straight into the parameter searches with each model.

**Note:** We will not test any stacking model and the reasoning behind this decision is two-fold. Firstly, generally stacking models are built based on the best models of parameter searches across single models (or boosting/bagging ensembles), since we are using full pipelines we can't just grab the best model, as that model depends on the choices made previously along the pipeline combination. Furthermore, we could develop a stacking classifier composed of best pipelines, however this would just create an explosion in complexity and run-time. All of this coupled with the fact that stacking models are always far more complex for often diminishing returns, leads us to believe that not testing stacking models is the best option. Also note that the parameter search code is in markdown cells, so it isn't re-run as it takes significant time, but results are imported to be analyzed using pickle files.

**Benchmarks**

As the goal of this project is to help the CSA create and improve a targeted approach to donors with the goal of maximizing donations, we will use 2 baseline benchmarks as the minimum any model must achieve to be a good model:
+ Random prediction, which in a dataset with the imbalance present in the CSA's data yield an **F1-Score of ~0.34**
+ Predicting all 1, which is essentially, telling the CSA to keep sending out the camapaign to everyone. This is obviously not the desire of the CSA and as such the goal is to create models that can avoid this. Predicting all 1 in a dataset with this imbalance yields an **F1-Score of 0.4**

**Table of contents**<a id='toc0_'></a>    
1. [Imports](#toc1_)    
2. [Defining the Pipeline](#toc2_)    
3. [Bagging Models](#toc3_)    
   3.1. [Random Forest](#toc3_1_)    
     3.1.1. [Parameter Search](#toc3_1_1_)    
     3.1.2. [Test Set Prediction](#toc3_1_2_)
4. [Boosting Models](#toc4_)    
   4.1. [Tree-Based AdaBoost](#toc4_1_)    
     4.1.1. [Parameter Search](#toc4_1_1_)   
     4.1.2. [Test Set Prediction](#toc4_1_2_)    
   4.2. [Gradient Boosting](#toc4_2_)   
     4.2.1. [Parameter Search](#toc4_2_1_)   
     4.2.2. [Test Set Prediction](#toc4_2_2_)

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=true
	minLevel=1
	maxLevel=3
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[Imports](#toc0_)

In this section we're importing everything we need, libraries and tools.

In [1]:
from utils_modeling import (OutlierClipper, CategoricalFeatureSelector, NumericalFeatureSelector, FeatureEngineer, DataCleaner, run_parameter_search)
import os
# Adding this ensures that the magic command only runs once (the first time)
if os.getcwd().split('\\')[-1] != 'DM2_Project':
    %cd ..
import numpy as np
import pandas as pd
from sklearn.model_selection import TunedThresholdClassifierCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler, MinMaxScaler, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.compose import make_column_selector
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, HistGradientBoostingClassifier, StackingClassifier
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import f1_score
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings("ignore")
SEED=23
pd.options.display.max_columns = None

c:\Users\dinis\OneDrive\Ambiente de Trabalho\Faculdade - MGI-BI\1º ano\2º Semestre\Data Mining II\Project\DM2_Project


In [2]:
train = pd.read_csv('Files/donors_train.csv')
test = pd.read_csv('Files/donors_test.csv')

X_train_preprocessed = pd.read_pickle('Files/Pickle Files/X_train_preprocessed.pkl')
X_val_preprocessed = pd.read_pickle('Files/Pickle Files/X_val_preprocessed.pkl')
y_train = pd.read_pickle('Files/Pickle Files/y_train.pkl')
y_val = pd.read_pickle('Files/Pickle Files/y_val.pkl')

with open('Files/Pickle Files/model_testing_skf.pkl', 'rb') as file:
    model_testing_skf = pickle.load(file)
with open('Files/Pickle Files/data_cleaner.pkl', 'rb') as file:
    data_cleaner = pickle.load(file)

In [3]:
X = train.drop('TARGET_B', axis=1)
y = train['TARGET_B']

# 2. <a id='toc2_'></a>[Defining the Pipeline](#toc0_)

We're first going to start by defining the pipeline we're gonna use as a base. This is the pipeline introduced in the Modeling Tools notebook, now with an ensemble model (a base RandomForest as a placeholder for now).

In [4]:
# Categorical Feature Sub-Pipeline
# This is the part that handles the categorical columns, performing
# mode imputation, feature selection using our custom CategoricalFeatureSelector
# and finally one-hot encoding the features.
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('feature_selection',  CategoricalFeatureSelector()),
    # 3. Your specialized encoding (OneHot/Target) now receives imputed integers
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')),
])

# Numerical Feature Sub-Pipeline
# Here we take care of our numerical features, starting with outlier clipping and feature
# creation using our custom transformer, then scaling data, so that it can then 
# be imputed and numerical feature selection can be performed
num_pipe = Pipeline([
    ('clipper', OutlierClipper()),
    ('feature_engineer', FeatureEngineer()),
    ('scaler', RobustScaler()),
    ('imputer', KNNImputer()),
    ('feature_selection', NumericalFeatureSelector(random_state=SEED))
])

# Here we use a ColumnTransformer with column selectors to perform the split
# between numerical and categorical data, so that each subset can be directed
# to the appropriate sub-pipeline
preprocessor = ColumnTransformer([
    ('cat_section', cat_pipe, make_column_selector(dtype_exclude=[np.number])),
    ('num_section', num_pipe, make_column_selector(dtype_include=[np.number])),
],
verbose_feature_names_out=False)
#  

 
# Final Pipeline
ensemble_pipe = Pipeline([
    ('cleaner', data_cleaner),
    ('preprocessing', preprocessor),
    ('model', TunedThresholdClassifierCV(RandomForestClassifier(),
                                         n_jobs=-1,
                                         scoring='f1',
                                         random_state=SEED,
                                         cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
                                         )) 
])

# We use the set_output to pandas so that intermediate transformers can use column
# names, since some of the default scikit-learn transformers by default return 
# Numpy arrays, which obviously don't have column names.
ensemble_pipe.set_output(transform="pandas")

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('cleaner', ...), ('preprocessing', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,categorical_cols_values,"{'DONOR_GENDER': ['M', 'F', ...], 'INCOME_GROUP': array([1, 2, 3, 4, 5, 6, 7]), 'PEP_STAR': [0, 1], 'RECENCY_STATUS_96NK': ['S', 'A', ...], ...}"
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat_section', ...), ('num_section', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that

Now let's start looking at models and results.

# 3. <a id='toc3_'></a>[Bagging Models](#toc0_)

## 3.1. <a id='toc3_1_'></a>[Random Forest](#toc0_)

A random forest is a bagging ensemble model which combines multiple decision trees, each using bootstrapped samples of data with a subset of the feature and its goal is combining many learners for more predictive performance. Due to this we can jump into the parameter search using some of the reference values we found in the DecisionTree modeling notebook.


### 3.1.1. <a id='toc3_1_1_'></a>[Parameter Search](#toc0_)

We're going to test several combinations across two separate sub-grids to optimize our preprocessing and modeling steps efficiently. This is because Random Forests don't require standard scaling, and so scalers will only be used when the KNNImputer is also used, except for the PowerTransformer which corrects variable skewness:

+ In preprocessing:
    + Scaler: For distance-based imputation, we are going to test the MinMaxScaler, RobustScaler (which uses quartile values to handle outliers smoothly), and finally the PowerTransformer which applies transformations to bring skewed distributions closer to a Normal distribution before scaling them to a mean of 0 and a variance of 1. For our iterative imputation path, we will skip the standard interval scalers since tree ensemble models do not require them, and we will only test the PowerTransformer alongside a baseline of no scaling at all.
    + Imputer: We will compare two advanced strategies: the distance-based KNNImputer set to 50 neighbors and the IterativeImputer. The Iterative Imputer uses a round-robin approach to model each missing feature as a function of all other features via its base BayesianRidge regression estimator, allowing us to see whether spatial proximity or multivariate regression handles our missing values best.
    + Clipping: Three outlier-handling approaches are tested across the combinations: no clipping at all (None), and clipping based on the standard IQR method for both normal and extreme outliers following the normal convention of 1.5 and 3 times the IQR beyond the quartiles, respectively.

+ In the model:
    + Tree Depth: We are tuning the max_depth parameter across three levels: 4, 6, and 10. While single decision trees require highly restrictive shallow depths to avoid overfitting, Random Forests can safely handle deeper trees because the ensemble averaging process naturally controls model variance and some overfiiting trees are often not a problem and beneficial for results.
    + Forest Size: We are testing ensemble sizes of 50 to 200, in steps of 50 trees to evaluate the performance plateau of the model against our computational runtime constraints.
    + Feature Subsampling: We are evaluating both sqrt and log2 strategies for the max_features parameter. This controls the size of the random feature subsets evaluated at each split point, forcing individual trees within the forest to remain highly decorrelated.

In [5]:
rf_param_grid = [
    # Sub-grid for KNN Imputer 
    {
        'preprocessing__num_section__imputer' : [KNNImputer(n_neighbors=50)],
        'preprocessing__num_section__scaler' : [PowerTransformer(), MinMaxScaler(), RobustScaler()], 
        'preprocessing__num_section__clipper' : [None, OutlierClipper(method='iqr', iqr_multiplier=1.5), OutlierClipper(method='iqr', iqr_multiplier=3)],
        'model__estimator' : [RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1)],
        'model__estimator__max_depth' : [4, 6, 10],
        'model__estimator__n_estimators' : [50, 100, 150, 200],
        'model__estimator__max_features' : ['sqrt', 'log2']
    },
    
    # Sub-grid for Iterative Imputer
    {
        'preprocessing__num_section__imputer' : [IterativeImputer(random_state=SEED, max_iter=10, initial_strategy='median')],
        'preprocessing__num_section__scaler' : [None, PowerTransformer()], 
        'preprocessing__num_section__clipper' : [None, OutlierClipper(method='iqr', iqr_multiplier=1.5), OutlierClipper(method='iqr', iqr_multiplier=3)],
        'model__estimator' : [RandomForestClassifier(class_weight='balanced', random_state=SEED, n_jobs=-1)],
        'model__estimator__max_depth' : [4, 6, 10], 
        'model__estimator__n_estimators' : [50, 100, 150, 200],
        'model__estimator__max_features' : ['sqrt', 'log2']
    }
]

```python
run_parameter_search(grid=rf_param_grid,
                     cv=model_testing_skf, 
                     X=X, y=y,
                     model=ensemble_pipe,
                     metrics=['f1', 'precision', 'recall'],
                     results_file_dir='Files/Pickle Files/Results/RF_GridSearch_Results.pkl',
                     model_file_dir='Files/Pickle Files/Pipelines/RF_GridSearch_Best_Pipeline.pkl',
                     refit=True,
                     n_jobs=-1)

In [6]:
rf_result_df = pd.read_pickle('Files/Pickle Files/Results/RF_GridSearch_Results.pkl')
rf_result_df.head()

,params_config,model__estimator,model__estimator__max_depth,model__estimator__max_features,model__estimator__n_estimators,preprocessing__num_section__clipper,preprocessing__num_section__imputer,preprocessing__num_section__scaler,mean_fit_time,mean_val_f1,std_val_f1,mean_train_f1,std_train_f1,mean_val_precision,std_val_precision,mean_train_precision,std_train_precision,mean_val_recall,std_val_recall,mean_train_recall,std_train_recall,status
125,{'model__estimator': RandomForestClassifier(cl...,RandomForestClassifier(class_weight='balanced'...,6,log2,100,OutlierClipper(iqr_multiplier=3),KNNImputer(n_neighbors=50),RobustScaler(),69.068055,0.421884,0.008525,0.461440,0.005948,0.289578,0.010960,0.317699,0.009656,0.779646,0.028457,0.846091,0.038424,Success
75,{'model__estimator': RandomForestClassifier(cl...,RandomForestClassifier(class_weight='balanced'...,6,sqrt,50,OutlierClipper(),KNNImputer(n_neighbors=50),PowerTransformer(),77.491940,0.421557,0.006330,0.465330,0.006629,0.290491,0.003355,0.321363,0.008461,0.769322,0.035010,0.844248,0.017216,Success
89,{'model__estimator': RandomForestClassifier(cl...,RandomForestClassifier(class_weight='balanced'...,6,sqrt,100,OutlierClipper(iqr_multiplier=3),KNNImputer(n_neighbors=50),RobustScaler(),69.728558,0.421001,0.006914,0.465796,0.006719,0.290568,0.005601,0.322102,0.009994,0.766077,0.040635,0.843215,0.025465,Success
303,{'model__estimator': RandomForestClassifier(cl...,RandomForestClassifier(class_weight='balanced'...,6,log2,150,OutlierClipper(),"IterativeImputer(initial_strategy='median', ra...",PowerTransformer(),160.557132,0.420619,0.004381,0.465749,0.008500,0.291754,0.005324,0.323355,0.011858,0.755457,0.036943,0.835029,0.026709,Success
110,{'model__estimator': RandomForestClassifier(cl...,RandomForestClassifier(class_weight='balanced'...,6,log2,50,None,KNNImputer(n_neighbors=50),RobustScaler(),73.665870,0.420255,0.006678,0.463204,0.006744,0.291620,0.004226,0.322285,0.010117,0.754572,0.046703,0.825516,0.027518,Success


In [7]:
rf_result_df.iloc[0]['params_config']

{'model__estimator': RandomForestClassifier(class_weight='balanced', max_depth=10,
                        max_features='log2', n_estimators=200, n_jobs=-1,
                        random_state=23),
 'model__estimator__max_depth': 6,
 'model__estimator__max_features': 'log2',
 'model__estimator__n_estimators': 100,
 'preprocessing__num_section__clipper': OutlierClipper(iqr_multiplier=3),
 'preprocessing__num_section__imputer': KNNImputer(n_neighbors=50),
 'preprocessing__num_section__scaler': RobustScaler()}

In [8]:
with open('Files/Pickle Files/Pipelines/RF_GridSearch_Best_Pipeline.pkl', 'rb') as file:
    rf_best_pipeline = pickle.load(file)

In [9]:
rf_best_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('cleaner', ...), ('preprocessing', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,categorical_cols_values,"{'DONOR_GENDER': ['M', 'F', ...], 'INCOME_GROUP': array([1, 2, 3, 4, 5, 6, 7]), 'PEP_STAR': [0, 1], 'RECENCY_STATUS_96NK': ['S', 'A', ...], ...}"
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat_section', ...), ('num_section', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that

In [10]:
rf_best_pipeline['model'].best_threshold_

np.float64(0.44620669079427855)

The best parameter combination achieved a mean validation F1-score of 0.421884 and a mean training F1-score of 0.461440 indicating a small amount overfitting, which is common across the highest scoring combinations. It used the RobustScaler, Outlier clipping using the IQR method for extreme outliers (multiplier 3) and the KNNImputer. The RandomForest contained 100 trees of depth 6 each receiving log2 of the number of features.

Additionally, it has an optimized decision threshold of ~0.45 which is very simlilar to default 0.5, thus meaning that for this model threshold optimization isn't as aggressive when compared to other models. Still, slightly more positive predictions are produced by the slightly lower threshold.

### 3.1.2. <a id='toc3_1_2_'></a>[Test Set Prediction](#toc0_)

The last thing to do before moving to the next model is to predict for the test set, for our deployment simulation. For that we'll generate the prediction and export it to a CSV file as instructed.

In [11]:
rf_pred_test = pd.DataFrame(rf_best_pipeline.predict(test), index=test['CONTROL_NUMBER'], columns=['TARGET_B'])
rf_pred_test

,TARGET_B
CONTROL_NUMBER,
122653,0
184239,1
5172,1
135377,1
62119,0
...,...
54438,0
122194,1
106603,1


In [12]:
rf_pred_test.to_csv('Files/Submissions/DM2DT_Group12_Version22.csv')

The best model from the RandomForest parameter search achieves an F1-Score of 0.43304 as the public score on Kaggle. We also add this score to the kaggle results file and then re-export it, with the cv scores directly from the dataframe.

In [13]:
kaggle_results_df = pd.read_pickle('Files/Pickle Files/Kaggle_scores.pkl')

In [14]:
kaggle_results_df.loc['Best RF Pipeline'] = [0.43304, 'DM2DT_Group12_Version22.csv', rf_result_df.iloc[0]['mean_val_f1'], rf_result_df.iloc[0]['mean_train_f1']]

In [15]:
kaggle_results_df

,Kaggle Public Score,Submission File Name,CV Mean Val F1,CV Mean Train F1
Best DT Pipeline,0.41582,DM2DT_Group12_Version19.csv,0.411342,0.415477
Best KNN Pipeline,0.42145,DM2DT_Group12_Version18.csv,0.410642,0.431648
Best NB Pipeline,0.42893,DM2DT_Group12_Version20.csv,0.419758,0.419638
Best LR Pipeline,0.43087,DM2DT_Group12_Version25.csv,0.421687,0.423902
Best RF Pipeline,0.43304,DM2DT_Group12_Version22.csv,0.421884,0.461440
Best AdaBoost Pipeline,0.43735,DM2DT_Group12_Version23.csv,0.412953,0.416364
Best GB Pipeline,0.43478,DM2DT_Group12_Version24.csv,0.420412,0.466219
Best NN Pipeline,0.43264,DM2DT_Group12_Version29.csv,0.418193,0.421301


In [16]:
kaggle_results_df.to_pickle('Files/Pickle Files/Kaggle_scores.pkl')

# 4. <a id='toc4_'></a>[Boosting Models](#toc0_)

## 4.1. <a id='toc4_1_'></a>[Tree-Based AdaBoost](#toc0_)
An tree-based AdaBoost model is a boosting ensemble model that sequentially combines multiple decision trees—typically shallow ones or "stumps" and where each subsequent tree focuses on correcting the errors of its predecessor by adjusting data weights. Its goal is to convert an ensemble of weak learners into a single strong predictive model. Due to this sequential dependency and simpler base estimators, our parameter search will shift focus away from deep tree architectures and instead target the ideal balance between the number of estimators and the learning rate.

### 4.1.1. <a id='toc4_1_1_'></a>[Parameter Search](#toc0_)

We're going to test several combinations across two separate sub-grids to optimize our preprocessing and modeling steps efficiently. This is because boosting models rely on sequential error-correction rather than parallel averaging, allowing us to evaluate whether a boosted ensemble of weak decision trees outperforms a standard parallel forest:

+ In preprocessing:
    + Scaler: For distance-based imputation, we are going to test the MinMaxScaler, RobustScaler (which uses quartile values to handle outliers smoothly), and finally the PowerTransformer which applies transformations to bring skewed distributions closer to a Normal distribution before scaling them to a mean of 0 and a variance of 1. For our iterative imputation path, we will skip the standard interval scalers since tree ensemble models do not require them, and we will only test the PowerTransformer alongside a baseline of no scaling at all.
    + Imputer: We will compare two advanced strategies: the distance-based KNNImputer set to 50 neighbors and the IterativeImputer. The Iterative Imputer uses a round-robin approach to model each missing feature as a function of all other features via its base BayesianRidge regression estimator, allowing us to see whether spatial proximity or multivariate regression handles our missing values best.
    + Clipping: Three outlier-handling approaches are tested across the combinations: no clipping at all (None), and clipping based on the standard IQR method for both normal and extreme outliers following the normal convention of 1.5 and 3 times the IQR beyond the quartiles, respectively.

+ In the model:
    + Base Estimator Depth: We are testing shallow decision tree base estimators with maximum depths of 1, 2, and 3. This allows us to see if AdaBoost performs better when combining ultra-simple decision stumps or slightly deeper trees that can capture slightly more complex, multi-variable feature interactions.
    + Learning Rate: We are tuning the learning rate across 0.01, 0.1, and 1.0 to precisely control the shrinkage of each successive tree's contribution and find the optimal step size for sequential optimization.
    + Ensemble Size: We are testing configurations of 50, 100, and 150 sequential estimators to evaluate model convergence while strictly staying within our computational runtime limits.

In [17]:
adaboost_param_grid = [
    # Sub-grid for KNN Imputer 
    {
        'preprocessing__num_section__imputer' : [KNNImputer(n_neighbors=50)],
        'preprocessing__num_section__scaler' : [PowerTransformer(), MinMaxScaler(), RobustScaler()], 
        'preprocessing__num_section__clipper' : [None, OutlierClipper(method='iqr', iqr_multiplier=1.5), OutlierClipper(method='iqr', iqr_multiplier=3)],
        'model__estimator' : [AdaBoostClassifier(random_state=SEED)],
        'model__estimator__estimator' : [
            DecisionTreeClassifier(max_depth=1, class_weight='balanced', random_state=SEED), 
            DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=SEED),
            DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=SEED)
        ],
        'model__estimator__n_estimators' : [50, 100, 150],
        'model__estimator__learning_rate' : [0.01, 0.1, 1.0] # Removed 0.05 to fix the budget
    },
    
    # Sub-grid for Iterative Imputer 
    {
        'preprocessing__num_section__imputer' : [IterativeImputer(random_state=SEED, max_iter=10, initial_strategy='median')],
        'preprocessing__num_section__scaler' : [None, PowerTransformer()], 
        'preprocessing__num_section__clipper' : [None, OutlierClipper(method='iqr', iqr_multiplier=1.5), OutlierClipper(method='iqr', iqr_multiplier=3)],
        'model__estimator' : [AdaBoostClassifier(random_state=SEED)],
        'model__estimator__estimator' : [
            DecisionTreeClassifier(max_depth=1, class_weight='balanced', random_state=SEED), 
            DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=SEED),
            DecisionTreeClassifier(max_depth=3, class_weight='balanced', random_state=SEED)
        ],
        'model__estimator__n_estimators' : [50, 100, 150],
        'model__estimator__learning_rate' : [0.01, 0.1, 1.0]
    }
]

```python
run_parameter_search(grid=adaboost_param_grid,
                     cv=model_testing_skf, 
                     X=X, y=y,
                     model=ensemble_pipe,
                     metrics=['f1', 'precision', 'recall'],
                     results_file_dir='Files/Pickle Files/Results/AdaBoost_GridSearch_Results.pkl',
                     model_file_dir='Files/Pickle Files/Pipelines/AdaBoost_GridSearch_Best_Pipeline.pkl',
                     refit=True,
                     n_jobs=-1)

In [18]:
ab_result_df = pd.read_pickle('Files/Pickle Files/Results/AdaBoost_GridSearch_Results.pkl')
ab_result_df.head()

,params_config,model__estimator,model__estimator__estimator,model__estimator__learning_rate,model__estimator__n_estimators,preprocessing__num_section__clipper,preprocessing__num_section__imputer,preprocessing__num_section__scaler,mean_fit_time,mean_val_f1,std_val_f1,mean_train_f1,std_train_f1,mean_val_precision,std_val_precision,mean_train_precision,std_train_precision,mean_val_recall,std_val_recall,mean_train_recall,std_train_recall,status
183,{'model__estimator': AdaBoostClassifier(estima...,AdaBoostClassifier(estimator=DecisionTreeClass...,DecisionTreeClassifier(class_weight='balanced'...,0.01,150,OutlierClipper(),KNNImputer(n_neighbors=50),PowerTransformer(),85.820642,0.412953,0.007928,0.416364,0.008266,0.273427,0.011842,0.275861,0.013198,0.853392,0.076370,0.859071,0.072226,Success
174,{'model__estimator': AdaBoostClassifier(estima...,AdaBoostClassifier(estimator=DecisionTreeClass...,DecisionTreeClassifier(class_weight='balanced'...,0.01,100,OutlierClipper(),KNNImputer(n_neighbors=50),PowerTransformer(),83.804795,0.412953,0.007928,0.416364,0.008266,0.273427,0.011842,0.275861,0.013198,0.853392,0.076370,0.859071,0.072226,Success
386,{'model__estimator': AdaBoostClassifier(estima...,AdaBoostClassifier(estimator=DecisionTreeClass...,DecisionTreeClassifier(class_weight='balanced'...,0.10,150,OutlierClipper(iqr_multiplier=3),"IterativeImputer(initial_strategy='median', ra...",PowerTransformer(),135.974692,0.411230,0.006390,0.412611,0.006781,0.276772,0.021889,0.276753,0.019563,0.834808,0.126517,0.845354,0.131469,Success
380,{'model__estimator': AdaBoostClassifier(estima...,AdaBoostClassifier(estimator=DecisionTreeClass...,DecisionTreeClassifier(class_weight='balanced'...,0.10,100,OutlierClipper(iqr_multiplier=3),"IterativeImputer(initial_strategy='median', ra...",PowerTransformer(),97.605605,0.411230,0.006390,0.412611,0.006781,0.276772,0.021889,0.276753,0.019563,0.834808,0.126517,0.845354,0.131469,Success
374,{'model__estimator': AdaBoostClassifier(estima...,AdaBoostClassifier(estimator=DecisionTreeClass...,DecisionTreeClassifier(class_weight='balanced'...,0.10,50,OutlierClipper(iqr_multiplier=3),"IterativeImputer(initial_strategy='median', ra...",PowerTransformer(),133.969752,0.411230,0.006390,0.412611,0.006781,0.276772,0.021889,0.276753,0.019563,0.834808,0.126517,0.845354,0.131469,Success


In [19]:
ab_result_df.iloc[0]['params_config']

{'model__estimator': AdaBoostClassifier(estimator=DecisionTreeClassifier(class_weight='balanced',
                                                     max_depth=3,
                                                     random_state=23),
                    n_estimators=150, random_state=23),
 'model__estimator__estimator': DecisionTreeClassifier(class_weight='balanced', max_depth=3, random_state=23),
 'model__estimator__learning_rate': 0.01,
 'model__estimator__n_estimators': 150,
 'preprocessing__num_section__clipper': OutlierClipper(),
 'preprocessing__num_section__imputer': KNNImputer(n_neighbors=50),
 'preprocessing__num_section__scaler': PowerTransformer()}

In [20]:
ab_result_df.iloc[1]['params_config']

{'model__estimator': AdaBoostClassifier(estimator=DecisionTreeClassifier(class_weight='balanced',
                                                     max_depth=3,
                                                     random_state=23),
                    n_estimators=150, random_state=23),
 'model__estimator__estimator': DecisionTreeClassifier(class_weight='balanced', max_depth=3, random_state=23),
 'model__estimator__learning_rate': 0.01,
 'model__estimator__n_estimators': 100,
 'preprocessing__num_section__clipper': OutlierClipper(),
 'preprocessing__num_section__imputer': KNNImputer(n_neighbors=50),
 'preprocessing__num_section__scaler': PowerTransformer()}

The best parameter combination achieved a mean validation F1-score of 0.412953 and a mean training F1-score of 0.416364 indicating basically no overfitting. It used the PowerTransformer, Outlier clipping using the IQR method for normal outliers (multiplier 1.5) and the KNNImputer. The model had a learning rate of 0.01 and contained contained 150 trees of depth 3. However a model with otherwise identical parameters but 100 estimators instead of 150 achieved the exact same score and, since it's slightly less computationally complex we will actually consider this model as our best one.

```python
ab_bestmodel = clone(ensemble_pipe)
ab_bestmodel = ab_bestmodel.set_params(**ab_result_df.iloc[1]['params_config'])
ab_bestmodel.set_output(transform="pandas")
ab_bestmodel.fit(X, y)
with open('Files/Pickle Files/Pipelines/AdaBoost_GridSearch_Best_Pipeline.pkl', 'wb') as file:
                pickle.dump(ab_bestmodel, file)

In [21]:
with open('Files/Pickle Files/Pipelines/AdaBoost_GridSearch_Best_Pipeline.pkl', 'rb') as file:
    ab_best_pipeline = pickle.load(file)

In [22]:
ab_best_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('cleaner', ...), ('preprocessing', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,categorical_cols_values,"{'DONOR_GENDER': ['M', 'F', ...], 'INCOME_GROUP': array([1, 2, 3, 4, 5, 6, 7]), 'PEP_STAR': [0, 1], 'RECENCY_STATUS_96NK': ['S', 'A', ...], ...}"
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat_section', ...), ('num_section', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that

In [23]:
ab_best_pipeline['model'].best_threshold_

np.float64(0.14228153280865585)

The best pipeline it has an optimized decision threshold of ~0.14 which is significantly lower than the default 0.5, thus meaning that a lot more observations receive positive predictions due to threshold optimization, which means we "contact" more donors, but we also find more actual donors (higher recall), this is in our opinion a worthy compromise.

### 4.1.2. <a id='toc4_1_2_'></a>[Test Set Prediction](#toc0_)

The last thing to do before moving to the final model is to predict for the test set, for our deployment simulation. For that we'll generate the prediction and export it to a CSV file as instructed.

In [24]:
ab_pred_test = pd.DataFrame(ab_best_pipeline.predict(test), index=test['CONTROL_NUMBER'], columns=['TARGET_B'])
ab_pred_test

,TARGET_B
CONTROL_NUMBER,
122653,0
184239,1
5172,1
135377,1
62119,0
...,...
54438,0
122194,1
106603,1


In [25]:
ab_pred_test.to_csv('Files/Submissions/DM2DT_Group12_Version23.csv')

The best model from the AdaBoost parameter search achieves an F1-Score of 0.43735 as the public score on Kaggle. We also add this score to the kaggle results file and then re-export it, with the cv scores directly from the dataframe.

In [26]:
# We re-read and re-export the file so that the 3 model sections of this notebook can be executed indepently
kaggle_results_df = pd.read_pickle('Files/Pickle Files/Kaggle_scores.pkl')

In [27]:
# We use iloc[1] because we're actually considering our second best performing model as the best on
# It's not actually relevant since they score the same, but this is more correct
kaggle_results_df.loc['Best AdaBoost Pipeline'] = [0.43735, 'DM2DT_Group12_Version23.csv', ab_result_df.iloc[1]['mean_val_f1'], ab_result_df.iloc[1]['mean_train_f1']]

In [28]:
kaggle_results_df

,Kaggle Public Score,Submission File Name,CV Mean Val F1,CV Mean Train F1
Best DT Pipeline,0.41582,DM2DT_Group12_Version19.csv,0.411342,0.415477
Best KNN Pipeline,0.42145,DM2DT_Group12_Version18.csv,0.410642,0.431648
Best NB Pipeline,0.42893,DM2DT_Group12_Version20.csv,0.419758,0.419638
Best LR Pipeline,0.43087,DM2DT_Group12_Version25.csv,0.421687,0.423902
Best RF Pipeline,0.43304,DM2DT_Group12_Version22.csv,0.421884,0.461440
Best AdaBoost Pipeline,0.43735,DM2DT_Group12_Version23.csv,0.412953,0.416364
Best GB Pipeline,0.43478,DM2DT_Group12_Version24.csv,0.420412,0.466219
Best NN Pipeline,0.43264,DM2DT_Group12_Version29.csv,0.418193,0.421301


In [29]:
kaggle_results_df.to_pickle('Files/Pickle Files/Kaggle_scores.pkl')

## 4.2. <a id='toc4_2_'></a>[Gradient Boosting](#toc0_)

A Gradient Boosting classifier is a boosting ensemble model that sequentially builds multiple decision trees, where each new tree is trained to predict the residual errors (the differences between the actual and predicted values) of the previous trees combined. Its goal is to minimize a loss function by iteratively adding learners that point in the direction of steepest descent. Due to this architecture, we can leverage our previous insights from the DecisionTree notebook to set a baseline for tree depth, but our primary parameter search will now focus on tuning the interaction between the learning rate and the total number of trees to prevent overfitting.

**Note:** We are using HistGradientBoostingClassifier over the regular GradientBoostingClassifier, because according to sklearn's documentation it is much faster for datasets over 10000 rows.

### 4.2.1. <a id='toc4_2_1_'></a>[Parameter Search](#toc0_)

We're going to test several combinations across two separate sub-grids to optimize our preprocessing and modeling steps efficiently:

+ In preprocessing:
    + Scaler: For distance-based imputation, we are going to test the MinMaxScaler, RobustScaler (which uses quartile values to handle outliers smoothly), and finally the PowerTransformer which applies transformations to bring skewed distributions closer to a Normal distribution before scaling them to a mean of 0 and a variance of 1. For our iterative path, we will skip the standard interval scalers since tree ensemble models do not require them, and we will only test the PowerTransformer alongside a baseline of no scaling at all.
    + Imputer: We will compare two separate strategies: the distance-based KNNImputer set to 50 neighbors, and the IterativeImputer configured with a median initial strategy and a maximum of 10 iterations.
    + Clipping: Three outlier-handling approaches are tested universally across both sub-grid paths: no clipping at all (None), and clipping based on the standard IQR method for both normal and extreme outliers following the normal convention of 1.5 and 3 times the IQR beyond the quartiles, respectively.

+ In the model:
    + Tree Depth: We are tuning the max_depth parameter across four structural levels: 3, 5, 8, and 12. This allows the model to evaluate very shallow interaction paths up to highly complex relationships between features.
    + Learning Rate: We are evaluating learning rates of 0.01, 0.05, and 0.1 to establish the exact step size and shrinkage applied to each sequential tree, balancing fast convergence against the risk of overshooting the global minimum loss.
    + Boosting Iterations: We are testing max_iter bounds of 100 and 150 trees to find the optimal ensemble size threshold before performance gains plateau

In [30]:
gb_param_grid = [
    # Sub-grid for KNN Imputer
    {
        'preprocessing__num_section__imputer' : [KNNImputer(n_neighbors=50)],
        'preprocessing__num_section__scaler' : [PowerTransformer(), MinMaxScaler(), RobustScaler()], 
        'preprocessing__num_section__clipper' : [None, OutlierClipper(method='iqr', iqr_multiplier=1.5), OutlierClipper(method='iqr', iqr_multiplier=3)],
        'model__estimator' : [HistGradientBoostingClassifier(class_weight='balanced', random_state=SEED)],
        'model__estimator__max_depth' : [3, 5, 8, 12],   
        'model__estimator__learning_rate' : [0.01, 0.05, 0.1], 
        'model__estimator__max_iter' : [100, 150]           
    },
    
    # Sub-grid for Iterative Imputer
    {
        'preprocessing__num_section__imputer' : [IterativeImputer(random_state=SEED, max_iter=10, initial_strategy='median')],
        'preprocessing__num_section__scaler' : [None, PowerTransformer()], 
        'preprocessing__num_section__clipper' : [None, OutlierClipper(method='iqr', iqr_multiplier=1.5), OutlierClipper(method='iqr', iqr_multiplier=3)],
        'model__estimator' : [HistGradientBoostingClassifier(class_weight='balanced', random_state=SEED)],
        'model__estimator__max_depth' : [3, 5, 8, 12],         # Expanded to 4 options
        'model__estimator__learning_rate' : [0.01, 0.05, 0.1],
        'model__estimator__max_iter' : [100, 150]
    }
]

```python
run_parameter_search(grid=gb_param_grid,
                     cv=model_testing_skf, 
                     X=X, y=y,
                     model=ensemble_pipe,
                     metrics=['f1', 'precision', 'recall'],
                     results_file_dir='Files/Pickle Files/Results/GB_GridSearch_Results.pkl',
                     model_file_dir='Files/Pickle Files/Pipelines/GB_GridSearch_Best_Pipeline.pkl',
                     refit=True,
                     n_jobs=-1)

In [31]:
gb_result_df = pd.read_pickle('Files/Pickle Files/Results/GB_GridSearch_Results.pkl')
gb_result_df.head()

,params_config,model__estimator,model__estimator__learning_rate,model__estimator__max_depth,model__estimator__max_iter,preprocessing__num_section__clipper,preprocessing__num_section__imputer,preprocessing__num_section__scaler,mean_fit_time,mean_val_f1,std_val_f1,mean_train_f1,std_train_f1,mean_val_precision,std_val_precision,mean_train_precision,std_train_precision,mean_val_recall,std_val_recall,mean_train_recall,std_train_recall,status
34,{'model__estimator': HistGradientBoostingClass...,HistGradientBoostingClassifier(class_weight='b...,0.01,5,150,OutlierClipper(iqr_multiplier=3),KNNImputer(n_neighbors=50),MinMaxScaler(),87.427648,0.420412,0.005149,0.466219,0.002786,0.289115,0.004062,0.320928,0.003052,0.770501,0.016038,0.852286,0.016160,Success
65,{'model__estimator': HistGradientBoostingClass...,HistGradientBoostingClassifier(class_weight='b...,0.01,12,150,None,KNNImputer(n_neighbors=50),RobustScaler(),81.812015,0.419373,0.005991,0.485784,0.008661,0.287735,0.004573,0.333788,0.009794,0.774926,0.038784,0.893215,0.014240,Success
47,{'model__estimator': HistGradientBoostingClass...,HistGradientBoostingClassifier(class_weight='b...,0.01,8,150,None,KNNImputer(n_neighbors=50),RobustScaler(),51.248571,0.419359,0.005628,0.474371,0.007005,0.282442,0.004796,0.320502,0.008267,0.815044,0.029345,0.913717,0.017323,Success
79,{'model__estimator': HistGradientBoostingClass...,HistGradientBoostingClassifier(class_weight='b...,0.05,3,100,OutlierClipper(iqr_multiplier=3),KNNImputer(n_neighbors=50),MinMaxScaler(),78.425979,0.418738,0.003697,0.449623,0.007612,0.290821,0.008174,0.312463,0.014469,0.753687,0.060598,0.808850,0.056173,Success
88,{'model__estimator': HistGradientBoostingClass...,HistGradientBoostingClassifier(class_weight='b...,0.05,3,150,OutlierClipper(iqr_multiplier=3),KNNImputer(n_neighbors=50),MinMaxScaler(),79.638063,0.418246,0.005280,0.447720,0.006275,0.285869,0.005192,0.306126,0.009203,0.781416,0.041038,0.836062,0.037521,Success


In [32]:
gb_result_df.iloc[0]['params_config']

{'model__estimator': HistGradientBoostingClassifier(class_weight='balanced', max_depth=12,
                                max_iter=150, random_state=23),
 'model__estimator__learning_rate': 0.01,
 'model__estimator__max_depth': 5,
 'model__estimator__max_iter': 150,
 'preprocessing__num_section__clipper': OutlierClipper(iqr_multiplier=3),
 'preprocessing__num_section__imputer': KNNImputer(n_neighbors=50),
 'preprocessing__num_section__scaler': MinMaxScaler()}

In [33]:
with open('Files/Pickle Files/Pipelines/GB_GridSearch_Best_Pipeline.pkl', 'rb') as file:
    gb_best_pipeline = pickle.load(file)

In [34]:
gb_best_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('cleaner', ...), ('preprocessing', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,categorical_cols_values,"{'DONOR_GENDER': ['M', 'F', ...], 'INCOME_GROUP': array([1, 2, 3, 4, 5, 6, 7]), 'PEP_STAR': [0, 1], 'RECENCY_STATUS_96NK': ['S', 'A', ...], ...}"
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat_section', ...), ('num_section', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that

In [35]:
gb_best_pipeline['model'].best_threshold_

np.float64(0.46958462105329635)

The best parameter combination achieved a mean validation F1-score of 0.420412 and a mean training F1-score of 0.466219 indicating some of overfitting. It used the MinMax Scaler, Outlier clipping using the IQR method for extreme outliers (multiplier 3) and the KNNImputer. The model had a learning rate of 0.01, a maximum of 150 trees of depth 5. <p>
Additionally, it has an optimized decision threshold of ~0.47, meaning a very small change from the default threshold, meaning the threshold optimization added some more positive predictions.

### 4.2.2. <a id='toc4_2_2_'></a>[Test Set Prediction](#toc0_)

The last thing to do is to predict for the test set. For that we'll generate the prediction and export it to a CSV file as instructed.

In [36]:
gb_pred_test = pd.DataFrame(gb_best_pipeline.predict(test), index=test['CONTROL_NUMBER'], columns=['TARGET_B'])
gb_pred_test

,TARGET_B
CONTROL_NUMBER,
122653,0
184239,1
5172,1
135377,1
62119,0
...,...
54438,0
122194,0
106603,1


In [37]:
gb_pred_test.to_csv('Files/Submissions/DM2DT_Group12_Version24.csv')

The best model from the Gradient Boosting parameter search achieves an F1-Score of 0.43478 as the public score on Kaggle. We also add this score to the kaggle results file and then re-export it, with the cv scores directly from the dataframe.

In [38]:
# We re-read and re-export the file so that the 3 model sections of this notebook can be executed indepently
kaggle_results_df = pd.read_pickle('Files/Pickle Files/Kaggle_scores.pkl')

In [39]:
kaggle_results_df.loc['Best GB Pipeline'] = [0.43478, 'DM2DT_Group12_Version24.csv', gb_result_df.iloc[0]['mean_val_f1'], gb_result_df.iloc[0]['mean_train_f1']]

In [40]:
kaggle_results_df

,Kaggle Public Score,Submission File Name,CV Mean Val F1,CV Mean Train F1
Best DT Pipeline,0.41582,DM2DT_Group12_Version19.csv,0.411342,0.415477
Best KNN Pipeline,0.42145,DM2DT_Group12_Version18.csv,0.410642,0.431648
Best NB Pipeline,0.42893,DM2DT_Group12_Version20.csv,0.419758,0.419638
Best LR Pipeline,0.43087,DM2DT_Group12_Version25.csv,0.421687,0.423902
Best RF Pipeline,0.43304,DM2DT_Group12_Version22.csv,0.421884,0.461440
Best AdaBoost Pipeline,0.43735,DM2DT_Group12_Version23.csv,0.412953,0.416364
Best GB Pipeline,0.43478,DM2DT_Group12_Version24.csv,0.420412,0.466219
Best NN Pipeline,0.43264,DM2DT_Group12_Version29.csv,0.418193,0.421301


In [41]:
kaggle_results_df.to_pickle('Files/Pickle Files/Kaggle_scores.pkl')